In [4]:
import pandas as pd
import yfinance as yf

# ============================
# Parameters
# ============================
START_DATE = "1980-01-01"
END_DATE = None  # None -> latest

# Yahoo tickers
TICKERS = {
    "VIX": "^VIX",         # CBOE Volatility Index
    "UST_10Y": "^TNX",     # 10-year Treasury yield index (x10)
    "DX_FUT": "DX=F",      # Dollar index futures (front contract)
}

# ============================
# Helper: robustly get Adj Close as a Series
# ============================
def download_series_yahoo(ticker, name, start=START_DATE, end=END_DATE):
    df = yf.download(ticker, start=start, end=end, auto_adjust=False)

    if df.empty:
        raise ValueError(f"No data returned for {ticker}")

    # df["Adj Close"] can be a Series or a DataFrame (if MultiIndex columns)
    adj = df["Adj Close"]

    # If it's a DataFrame (MultiIndex columns), take the first column
    if isinstance(adj, pd.DataFrame):
        adj = adj.iloc[:, 0]

    # Make sure it's a Series with a nice name
    adj = adj.astype("float64")
    adj.name = name
    return adj


# ============================
# 1) Yahoo Finance data
# ============================
# VIX
vix = download_series_yahoo(TICKERS["VIX"], "VIX")

# Treasury Yields
y10_raw = download_series_yahoo(TICKERS["UST_10Y"], "UST10Y_raw")

# Yahoo yields quoted as "yield * 10" (e.g. 46.50 = 4.65%)
y10 = (y10_raw / 10.0).rename("UST10Y_pct")

# Dollar index futures
dx_fut = download_series_yahoo(TICKERS["DX_FUT"], "DX_FUT")


# ============================
# 2) Combine everything
# ============================
macro = pd.concat(
    [
        vix,
        y10,
        dx_fut,
    ],
    axis=1
).sort_index()

macro = macro.loc[macro.index >= pd.to_datetime(START_DATE)]
macro = macro.dropna(how="all")

# Save
macro.to_csv("macro_yahoo_vix_yieldcurve_dxy.csv")

print(macro.tail())
print("\nSaved to macro_yahoo_vix_yieldcurve_dxy.csv")


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

                  VIX  UST10Y_pct  DX_FUT
Date                                     
2025-11-25  18.559999      0.4002     NaN
2025-11-26  17.190001      0.3998     NaN
2025-11-28  16.350000      0.4017     NaN
2025-12-01  17.240000      0.4096     NaN
2025-12-02  16.590000      0.4086   99.18

Saved to macro_yahoo_vix_yieldcurve_dxy.csv


In [5]:
df = pd.read_csv("macro_yahoo_vix_yieldcurve_dxy.csv", parse_dates=["Date"], index_col="Date")

df.tail()

,VIX,UST10Y_pct,DX_FUT
Date,,,
2025-11-25,18.559999,0.4002,NaN
2025-11-26,17.190001,0.3998,NaN
2025-11-28,16.350000,0.4017,NaN
2025-12-01,17.240000,0.4096,NaN
2025-12-02,16.590000,0.4086,99.18
